# Run Model

Single-country simulation runner with standard macro, benchmark, permanent-income, sector, and firm-credit diagnostics. Configure the inputs, run the workflow once, then inspect the manifest and diagnostic sections below.

## Companion notebooks

- `run_model_exploration.ipynb` — exploratory firm, household, reader, and ratio analysis
- `run_sensitivity.ipynb` — parameter sensitivity
- `run_mpc.ipynb` — household MPC experiment
- `run_irf.ipynb` — macro impulse responses
- `run_model_legacy_2026-08-18.ipynb` — preserved historical notebook; do not use for new work

In [1]:
# Setup
%load_ext autoreload
%autoreload 2

import numpy as np

from src.monte_carlo import run_seeded_monte_carlo
from config import (
    BALANCE_SHEET_COLUMNS,
    FIGURE_SIZES,
    FISCAL_COLUMNS,
    LABOUR_COLUMNS,
    MACRO_COLUMNS,
    POLICY_COLUMNS,
    SCENARIO_PRESETS,
)
from src.notebook_state import run_notebook_workflow, validate_notebook_state
from src.notebook_workflow import (
    NotebookRunConfig,
    build_permanent_income_forecast_contribution_table,
    build_permanent_income_log_ratio_decomposition_df,
    plot_permanent_income_log_ratio_decomposition,
)
from src.visual_helpers import (
    firm_sector_groups_table,
    plot_agent_timeseries,
    plot_cumulative_insolvent_firms_by_sector,
    plot_firm_credit_to_equity_and_capital,
    plot_mc,
    plot_output,
    plot_sector_tfp_investment_desired_mb_mc_ratio,
)

## Inputs

All execution switches are centralized here. Sensitivity, MPC, and IRF switches are in their dedicated notebooks.

In [2]:
RUN_BENCHMARK = True
RUN_MONTE_CARLO = False
SCENARIO_NAME = "calibrated_consumption"
# SCENARIO_NAME = "change_sectoral_weights"

run_config = NotebookRunConfig(
    seed=68,
    t_max=50,
    country_iso3="FRA",
    run_benchmark=RUN_BENCHMARK,
    force_rebuild_data=True,
    force_rerun_benchmark=True,
    benchmark_overrides=None,
)
scenario_overrides = SCENARIO_PRESETS[SCENARIO_NAME]




## Run simulation

In [3]:
# In the first run, state is none. After, run_notebook_workflow uses prepared from the previous state, preventing reloading it.
state = globals().get("state")

state = run_notebook_workflow(
    run_config,
    scenario_name=SCENARIO_NAME,
    scenario_overrides=scenario_overrides,
    previous_state=state,
)

# Familiar aliases; `state` remains the authoritative carrier.
COUNTRY = state.country_code
prepared = state.prepared
data = prepared.data
cfg = prepared.cfg
simulation = state.simulation
model = state.model
df_scenario = state.df_scenario
benchmark = state.benchmark
df_benchmark = state.df_benchmark

validate_notebook_state(state)

{'seed': 68, 'country': 'FRA', 't_max': 50, 'raw_data_path': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/raw_data', 'output_dir': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data', 'data_cache': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/data.pkl'}
Configuration summary
{'productivity_growth': 'SimpleTFPGrowth',
 'productivity_investment_planner': 'TargetIntensityTFPInvestmentPlanner',
 'labour_market': {'name': 'DefaultLabourMarketClearer',
                   'parameters': {'allow_switching_industries': True,
                                  'compare_with_normalised_inputs': True,
                                  'consider_reservation_wages': True,
                                  'firing_cost_fraction': 0.0,
                                  'firing_speed': 1.0,
                                  'hiring_cost_fraction': 0.0,
                                  'hiring_speed': 0.

/Users/andone/Documents/python_projects/INET-consumption/macromodel/agents/households/households_ts.py:270: RuntimeWarning: divide by zero encountered in divide
  rent_div_income_histogram=get_histogram(data["Rent Paid"].values / data["Income"].values, None),
/Users/andone/Documents/python_projects/INET-consumption/macromodel/agents/households/households_ts.py:270: RuntimeWarning: invalid value encountered in divide
  rent_div_income_histogram=get_histogram(data["Rent Paid"].values / data["Income"].values, None),


Simulation complete. Output saved to: /Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/simulation_FRA.h5
Configuration summary
{'productivity_growth': 'SimpleTFPGrowth',
 'productivity_investment_planner': 'TargetIntensityTFPInvestmentPlanner',
 'labour_market': {'name': 'DefaultLabourMarketClearer',
                   'parameters': {'allow_switching_industries': True,
                                  'compare_with_normalised_inputs': True,
                                  'consider_reservation_wages': True,
                                  'firing_cost_fraction': 0.0,
                                  'firing_speed': 1.0,
                                  'hiring_cost_fraction': 0.0,
                                  'hiring_speed': 0.75,
                                  'individuals_quitting': False,
                                  'individuals_quitting_temperature': 1.0,
                                  'optimised_hiring': True,
            

/Users/andone/Documents/python_projects/INET-consumption/macromodel/agents/households/households_ts.py:270: RuntimeWarning: divide by zero encountered in divide
  rent_div_income_histogram=get_histogram(data["Rent Paid"].values / data["Income"].values, None),
/Users/andone/Documents/python_projects/INET-consumption/macromodel/agents/households/households_ts.py:270: RuntimeWarning: invalid value encountered in divide
  rent_div_income_histogram=get_histogram(data["Rent Paid"].values / data["Income"].values, None),


Simulation complete. Output saved to: /Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/FRA_benchmark.h5
Benchmark dataframe cached at: /Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/FRA_df_benchmark.pkl


In [4]:
{
    "country": COUNTRY,
    "seed": cfg.seed,
    "t_max": cfg.t_max,
    "scenario": state.scenario_name,
    "model_h5": str(simulation.model_h5_path),
    "benchmark": benchmark is not None,
    "manifest": str(state.manifest_path),
}


{'country': 'FRA',
 'seed': 68,
 't_max': 50,
 'scenario': 'calibrated_consumption',
 'model_h5': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/simulation_FRA.h5',
 'benchmark': True,
 'manifest': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/run_manifest_FRA_seed68_t50.json'}

## Macro, fiscal, policy, labour, and balance sheets

In [5]:
REVENUES_COLS = [
    "debt_to_gdp",
    "deficit_to_gdp",
    "fiscal_revenue_to_gdp",
    "fiscal_expenditure_to_gdp",

    "fiscal_revenue_vat_to_gdp",
    "fiscal_revenue_production_taxes_to_gdp",
    "fiscal_revenue_capital_formation_taxes_to_gdp",
    "fiscal_revenue_corporate_income_taxes_to_gdp",
    "fiscal_revenue_income_taxes_to_gdp",
    "fiscal_revenue_rental_income_taxes_to_gdp",
    "fiscal_revenue_employee_social_insurance_to_gdp",
    "fiscal_revenue_employer_social_insurance_to_gdp",
    "fiscal_revenue_taxes_on_products_to_gdp",
    "fiscal_revenue_export_taxes_to_gdp",
    "fiscal_revenue_social_housing_rent_to_gdp",

    "fiscal_revenue_vat",
    "fiscal_revenue_production_taxes",
    "fiscal_revenue_capital_formation_taxes",
    "fiscal_revenue_corporate_income_taxes",
    "fiscal_revenue_income_taxes",
    "fiscal_revenue_rental_income_taxes",
    "fiscal_revenue_employee_social_insurance",
    "fiscal_revenue_employer_social_insurance",
    "fiscal_revenue_taxes_on_products",
    "fiscal_revenue_export_taxes",
    "fiscal_revenue_social_housing_rent",
    ]

EXPENDITURE_COLS = [
    "debt_to_gdp",
    "deficit_to_gdp",
    "fiscal_revenue_to_gdp",
    "fiscal_expenditure_to_gdp",

    "government_consumption_to_gdp",
    "unemployment_benefits_to_gdp",
    "interest_payments_on_debt_to_gdp",
    "household_social_transfers_to_gdp",
    "public_pension_benefits_to_gdp",
    "other_social_transfers_to_gdp",
    "necessity_support_to_gdp",

    "government_consumption",
    "unemployment_benefits",
    "interest_payments_on_debt",
    "household_social_transfers",
    "public_pension_benefits",
    "other_social_transfers",
    "necessity_support",
]


plot_output(df=df_scenario[list(REVENUES_COLS)], no_rows=8, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(EXPENDITURE_COLS)], no_rows=8, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


In [6]:
plot_output(
    df=df_scenario[list(MACRO_COLUMNS)],
    no_rows=5,
    no_cols=4,
    country_code=COUNTRY,
    line_color="#1f77b4",
    **FIGURE_SIZES["benchmark"],
)
plot_output(df=df_scenario[list(FISCAL_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(POLICY_COLUMNS)], no_rows=4, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(LABOUR_COLUMNS)], no_rows=2, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(BALANCE_SHEET_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


## Benchmark comparison

In [7]:
if df_benchmark is None:
    print("Benchmark disabled in Inputs.")
else:
    plot_output(df=df_scenario[list(MACRO_COLUMNS)], df_compare=df_benchmark[list(MACRO_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    plot_output(df=df_scenario[list(FISCAL_COLUMNS)], df_compare=df_benchmark[list(FISCAL_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    plot_output(df=df_scenario[list(POLICY_COLUMNS)], df_compare=df_benchmark[list(POLICY_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    plot_output(df=df_scenario[list(BALANCE_SHEET_COLUMNS)], df_compare=df_benchmark[list(BALANCE_SHEET_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


## Permanent-income decomposition

In [8]:
decomposition = build_permanent_income_log_ratio_decomposition_df(
    simulation,
    country_code=COUNTRY,
    reducer="mean",
    include_log_real_pc_income=True,
)
plot_permanent_income_log_ratio_decomposition(
    simulation,
    country_code=COUNTRY,
    columns=["ln_y_p_over_y", "common_log_ratio"],
    reducer="mean",
)

fig = plot_permanent_income_log_ratio_decomposition(
    simulation,
    country_code=COUNTRY,
    columns=["real_pc_income_idx"],
    reducer="mean",
    title='real_pc_income_idx',
    show=True,
)


contributions = build_permanent_income_forecast_contribution_table(
    simulation,
    country_code=COUNTRY,
    periods=list(range(9)),
    include_fixed=False,
)
contributions


,period,date,regressor,simulation_source,is_fixed,x_t,coefficient,contribution,point_forecast
0,0,2014Q1,time_trend,simulation_period_index,False,136.000000,0.004948,0.672896,0.022717
1,0,2014Q1,covid19,excluded_stage_3_dummy,False,0.000000,0.002059,0.000000,0.022717
2,0,2014Q1,log_real_pc_income,real_pc_income,False,4.605170,-1.017999,-4.688058,0.022717
3,0,2014Q1,d4_log_real_pc_income,real_pc_income,False,0.002550,0.068133,0.000174,0.022717
4,0,2014Q1,real_interest_rate_ma4_l1,policy_rate_and_cpi_fixed_basket,False,-0.757011,0.000638,-0.000483,0.022717
...,...,...,...,...,...,...,...,...,...
76,8,2016Q1,real_interest_rate_ma4_l1,policy_rate_and_cpi_fixed_basket,False,-1.588957,0.000638,-0.001013,0.055441
77,8,2016Q1,real_interest_rate_ma4_l5,policy_rate_and_cpi_fixed_basket,False,-0.487531,-0.002580,0.001258,0.055441
78,8,2016Q1,real_interest_rate_ma4_l9,policy_rate_and_cpi_fixed_basket,False,-0.996777,0.001678,-0.001673,0.055441
79,8,2016Q1,unemp_rate_ma4_l1,unemployment_rate,False,10.082251,-0.003852,-0.038837,0.055441


## Sector and firm-credit diagnostics

In [9]:
firm_sector_groups_table(model, COUNTRY)
plot_cumulative_insolvent_firms_by_sector(df_scenario)

credit_panels = [
    ["total_target_short_term_credit", "total_received_short_term_credit"],
    ["total_target_long_term_credit", "total_received_long_term_credit"],
    "short_term_loan_debt",
    "long_term_loan_debt",
]
plot_agent_timeseries(
    model,
    COUNTRY,
    "firms",
    variables=credit_panels,
    agg="sum",
    no_cols=2,
    show_legend=False,
    **FIGURE_SIZES["dense"],
)
plot_firm_credit_to_equity_and_capital(model, COUNTRY, show=True, return_df=False)
plot_sector_tfp_investment_desired_mb_mc_ratio(model, COUNTRY)


## Optional Monte Carlo

In [10]:
if RUN_MONTE_CARLO:
    rng = np.random.default_rng(run_config.seed)
    mc_seeds = rng.choice(np.arange(1000), size=50, replace=False).tolist()
    mc = run_seeded_monte_carlo(
        datawrapper=data,
        country_configurations=state.country_configurations,
        country_code=COUNTRY,
        seeds=mc_seeds,
        t_max=cfg.t_max,
        n_jobs=-1,
        backend="loky",
        batch_size=1,
    )
    plot_mc(mc=mc, cols=list(MACRO_COLUMNS), no_cols=4, country_code=COUNTRY)
else:
    print("Set RUN_MONTE_CARLO = True to run seeded Monte Carlo simulations.")


Set RUN_MONTE_CARLO = True to run seeded Monte Carlo simulations.
